# So sánh kết quả huấn luyện Faster R-CNN và RCNN-ODAM

Notebook này đọc trực tiếp artifact trong `results/`, ưu tiên các run hiện tại:

- `results/faster_rcnn`
- `results/rcnn_odam_train`

Nếu có thêm run cũ hoặc run thử nghiệm chứa `metrics.csv`, notebook vẫn tự phát hiện và đưa vào bảng inventory để tránh bỏ sót dữ liệu. Các output trong notebook được xóa sạch; hãy chạy lại toàn bộ notebook sau khi có kết quả train mới.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:  # pragma: no cover - only used outside notebooks
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string(index=False))
        else:
            print(obj)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
plt.style.use("seaborn-v0_8-whitegrid")


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for path in candidates:
        if (path / "results").exists() and (path / "rcnn_odamTrain").exists():
            return path
    for path in candidates:
        if (path / "results").exists():
            return path
    return start


PROJECT_ROOT = find_project_root()
RESULTS_ROOT = PROJECT_ROOT / "results"
REPORT_DIR = PROJECT_ROOT / "notebooks" / "compare_training_results_figures"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"RESULTS_ROOT = {RESULTS_ROOT}")
print(f"REPORT_DIR = {REPORT_DIR}")

## 1. Phát hiện run và kiểm tra artifact

Bảng này cho biết mỗi run có đủ `metrics.csv`, `test_metrics.json`, config và checkpoint hay không. Cột `preferred` đánh dấu các đường dẫn nên dùng cho so sánh công bằng sau khi sửa logic ODAM-Train.

In [ ]:
PREFERRED_RUN_CANDIDATES = [
    (
        "Faster R-CNN",
        [
            RESULTS_ROOT / "faster_rcnn",
            RESULTS_ROOT / "baseline" / "faster_rcnn_fixed",
            RESULTS_ROOT / "baseline" / "faster_rcnn",
            RESULTS_ROOT / "baseline" / "faster_rcnn_ddp",
        ],
    ),
    (
        "RCNN-ODAM",
        [
            RESULTS_ROOT / "rcnn_odam_train",
            RESULTS_ROOT / "rcnn_odam_train_fixed",
            RESULTS_ROOT / "rcnn_odam_train_ddp",
        ],
    ),
]


def rel(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT))
    except Exception:
        return str(path)


def read_json(path: Path | None) -> dict[str, Any]:
    if not path or not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def first_existing_config(run_dir: Path) -> Path | None:
    for name in ("run_config.json", "config.json", "args.json"):
        path = run_dir / name
        if path.exists():
            return path
    matches = sorted(run_dir.glob("*config*.json"))
    return matches[0] if matches else None


def checkpoint_summary(run_dir: Path) -> tuple[int, int, str]:
    checkpoints = sorted(run_dir.glob("*.pt")) + sorted(run_dir.glob("*.pth"))
    total_bytes = sum(p.stat().st_size for p in checkpoints if p.is_file())
    names = ", ".join(p.name for p in checkpoints[:4])
    if len(checkpoints) > 4:
        names += f", ... (+{len(checkpoints) - 4})"
    return len(checkpoints), total_bytes, names


def discover_runs() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    seen: set[Path] = set()

    for preferred_rank, (label, paths) in enumerate(PREFERRED_RUN_CANDIDATES):
        for candidate_rank, run_dir in enumerate(paths):
            run_dir = run_dir.resolve()
            if run_dir in seen:
                continue
            seen.add(run_dir)
            if not run_dir.exists():
                continue
            metrics_path = run_dir / "metrics.csv"
            test_metrics_path = run_dir / "test_metrics.json"
            config_path = first_existing_config(run_dir)
            n_ckpt, ckpt_bytes, ckpt_names = checkpoint_summary(run_dir)
            rows.append(
                {
                    "label": label if candidate_rank == 0 else f"{label} ({rel(run_dir)})",
                    "run_dir": run_dir,
                    "run_path": rel(run_dir),
                    "preferred": candidate_rank == 0,
                    "known_family_rank": preferred_rank,
                    "candidate_rank": candidate_rank,
                    "metrics_path": metrics_path if metrics_path.exists() else None,
                    "test_metrics_path": test_metrics_path if test_metrics_path.exists() else None,
                    "config_path": config_path,
                    "has_metrics": metrics_path.exists(),
                    "has_test_metrics": test_metrics_path.exists(),
                    "has_config": config_path is not None,
                    "checkpoint_count": n_ckpt,
                    "checkpoint_bytes_mb": round(ckpt_bytes / (1024 * 1024), 2),
                    "checkpoint_names": ckpt_names,
                }
            )

    for metrics_path in sorted(RESULTS_ROOT.glob("**/metrics.csv")):
        run_dir = metrics_path.parent.resolve()
        if run_dir in seen:
            continue
        seen.add(run_dir)
        test_metrics_path = run_dir / "test_metrics.json"
        config_path = first_existing_config(run_dir)
        n_ckpt, ckpt_bytes, ckpt_names = checkpoint_summary(run_dir)
        rows.append(
            {
                "label": rel(run_dir).replace("results/", ""),
                "run_dir": run_dir,
                "run_path": rel(run_dir),
                "preferred": False,
                "known_family_rank": 99,
                "candidate_rank": 99,
                "metrics_path": metrics_path,
                "test_metrics_path": test_metrics_path if test_metrics_path.exists() else None,
                "config_path": config_path,
                "has_metrics": True,
                "has_test_metrics": test_metrics_path.exists(),
                "has_config": config_path is not None,
                "checkpoint_count": n_ckpt,
                "checkpoint_bytes_mb": round(ckpt_bytes / (1024 * 1024), 2),
                "checkpoint_names": ckpt_names,
            }
        )

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values(["known_family_rank", "candidate_rank", "run_path"]).reset_index(drop=True)


RUN_INVENTORY = discover_runs()
if RUN_INVENTORY.empty:
    print(f"No training metrics found under {RESULTS_ROOT}")
else:
    display(
        RUN_INVENTORY[
            [
                "label",
                "run_path",
                "preferred",
                "has_metrics",
                "has_test_metrics",
                "has_config",
                "checkpoint_count",
                "checkpoint_bytes_mb",
                "checkpoint_names",
            ]
        ]
    )

RUNS = RUN_INVENTORY[RUN_INVENTORY["has_metrics"]].copy() if not RUN_INVENTORY.empty else RUN_INVENTORY

## 2. Chuẩn hóa metric theo epoch

Faster R-CNN và RCNN-ODAM đang ghi metric bằng tên cột khác nhau. Cell này gom chúng về cùng schema để vẽ chung mà không sửa file kết quả gốc.

In [ ]:
COLUMN_ALIASES = {
    "epoch": ["epoch"],
    "lr": ["lr", "learning_rate"],
    "train_loss_total": ["train_loss", "loss_total", "total_loss"],
    "valid_loss_total": ["valid_loss_total", "val_loss", "validation_loss"],
    "map_50_95": ["val_map_50_95", "map_50_95", "metrics/mAP50-95(B)", "map"],
    "map50": ["val_map50", "map50", "val_map_50", "metrics/mAP50(B)"],
    "map75": ["val_map75", "map75", "val_map_75"],
    "ar_1": ["val_ar_1", "ar_1"],
    "ar_10": ["val_ar_10", "ar_10"],
    "ar_100": ["val_ar_100", "ar_100"],
    "map_small": ["val_map_small", "map_small"],
    "map_medium": ["val_map_medium", "map_medium"],
    "map_large": ["val_map_large", "map_large"],
    "gt_total": ["val_gt_total", "gt_total"],
    "pred_total": ["val_pred_total", "pred_total"],
}


def pick_column(raw: pd.DataFrame, aliases: list[str]) -> str | None:
    exact = {c: c for c in raw.columns}
    lower = {c.lower(): c for c in raw.columns}
    for name in aliases:
        if name in exact:
            return exact[name]
        if name.lower() in lower:
            return lower[name.lower()]
    return None


def numeric_or_nan(raw: pd.DataFrame, column: str | None) -> pd.Series:
    if column is None:
        return pd.Series([np.nan] * len(raw), index=raw.index, dtype="float64")
    return pd.to_numeric(raw[column], errors="coerce")


def normalize_epoch_metrics(label: str, run_dir: Path, metrics_path: Path) -> pd.DataFrame:
    raw = pd.read_csv(metrics_path)
    out = pd.DataFrame(index=raw.index)
    out["run"] = label
    out["run_path"] = rel(run_dir)

    for canonical, aliases in COLUMN_ALIASES.items():
        out[canonical] = numeric_or_nan(raw, pick_column(raw, aliases))

    if out["epoch"].isna().all():
        out["epoch"] = np.arange(1, len(out) + 1)
    out["epoch"] = out["epoch"].astype("Int64")

    for column in raw.columns:
        normalized = column.lower()
        if normalized.startswith("val_"):
            normalized = normalized[4:]
        if normalized.startswith("test_"):
            normalized = normalized[5:]
        if normalized.startswith("class_") and normalized.endswith("_ap50"):
            out[normalized] = pd.to_numeric(raw[column], errors="coerce")

    return out


metric_frames = []
for row in RUNS.itertuples(index=False):
    metric_frames.append(normalize_epoch_metrics(row.label, row.run_dir, row.metrics_path))

METRICS_DF = pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame()
if METRICS_DF.empty:
    print("No epoch metrics available.")
else:
    display(METRICS_DF.head())

## 3. Best epoch và test metrics

`best_epoch` được chọn theo thứ tự ưu tiên: validation `mAP50`, rồi `mAP50-95`, rồi epoch cuối nếu thiếu cả hai metric. Test metric được đọc từ cả hai dạng JSON: `metrics` của baseline và `coco_metrics`/`loss_metrics` của RCNN-ODAM.

In [ ]:
def best_epoch_summary(metrics_df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    if metrics_df.empty:
        return pd.DataFrame(rows)

    for run, df in metrics_df.groupby("run", sort=False):
        df = df.sort_values("epoch")
        metric_used = "last_epoch"
        if df["map50"].notna().any():
            idx = df["map50"].idxmax()
            metric_used = "max_val_map50"
        elif df["map_50_95"].notna().any():
            idx = df["map_50_95"].idxmax()
            metric_used = "max_val_map_50_95"
        else:
            idx = df.index[-1]
        row = df.loc[idx]
        rows.append(
            {
                "run": run,
                "run_path": row["run_path"],
                "best_epoch": int(row["epoch"]),
                "selection_metric": metric_used,
                "val_map50": row["map50"],
                "val_map_50_95": row["map_50_95"],
                "val_map75": row["map75"],
                "val_ar_100": row["ar_100"],
                "train_loss_total": row["train_loss_total"],
                "valid_loss_total": row["valid_loss_total"],
                "val_pred_total": row["pred_total"],
                "val_gt_total": row["gt_total"],
            }
        )
    return pd.DataFrame(rows)


def normalize_test_metrics(label: str, run_dir: Path, test_metrics_path: Path | None) -> dict[str, Any]:
    obj = read_json(test_metrics_path)
    coco_metrics = obj.get("metrics") or obj.get("coco_metrics") or {}
    loss_metrics = obj.get("loss_metrics") or {}
    row: dict[str, Any] = {
        "run": label,
        "run_path": rel(run_dir),
        "test_metrics_path": rel(test_metrics_path) if test_metrics_path else None,
        "test_checkpoint": obj.get("checkpoint"),
        "test_checkpoint_epoch": obj.get("checkpoint_epoch"),
        "test_images": obj.get("images"),
        "test_instances": obj.get("instances"),
    }
    for name in [
        "map_50_95",
        "map50",
        "map75",
        "map_small",
        "map_medium",
        "map_large",
        "ar_1",
        "ar_10",
        "ar_100",
        "ar_small",
        "ar_medium",
        "ar_large",
        "gt_total",
        "pred_total",
    ]:
        row[f"test_{name}"] = coco_metrics.get(name, np.nan)
    row["test_loss_total"] = loss_metrics.get("test_loss_total", np.nan)

    for key, value in coco_metrics.items():
        normalized = key.lower()
        if normalized.startswith("class_") and normalized.endswith("_ap50"):
            row[f"test_{normalized}"] = value
    return row


BEST_EPOCH_DF = best_epoch_summary(METRICS_DF)
TEST_METRICS_DF = pd.DataFrame(
    [normalize_test_metrics(row.label, row.run_dir, row.test_metrics_path) for row in RUNS.itertuples(index=False)]
)
SUMMARY_DF = BEST_EPOCH_DF.merge(TEST_METRICS_DF, on=["run", "run_path"], how="outer") if not BEST_EPOCH_DF.empty else TEST_METRICS_DF

if SUMMARY_DF.empty:
    print("No summary available.")
else:
    summary_columns = [
        "run",
        "run_path",
        "best_epoch",
        "selection_metric",
        "val_map50",
        "val_map_50_95",
        "test_map50",
        "test_map_50_95",
        "test_map75",
        "test_ar_100",
        "test_pred_total",
        "test_gt_total",
        "test_loss_total",
        "test_checkpoint_epoch",
    ]
    display(SUMMARY_DF[[c for c in summary_columns if c in SUMMARY_DF.columns]])

## 4. Kiểm tra điều kiện so sánh công bằng

Cell này không kết luận mô hình nào tốt hơn; nó chỉ chỉ ra các khác biệt cấu hình dễ làm lệch so sánh như pretrained weights, learning rate, epoch, batch size, image size hoặc thiếu test metrics.

In [ ]:
def nested_get(obj: dict[str, Any], *paths: str, default: Any = np.nan) -> Any:
    for path in paths:
        cur: Any = obj
        ok = True
        for part in path.split("."):
            if isinstance(cur, dict) and part in cur:
                cur = cur[part]
            else:
                ok = False
                break
        if ok:
            return cur
    return default


def config_row(row: Any) -> dict[str, Any]:
    cfg = read_json(row.config_path)
    return {
        "run": row.label,
        "run_path": row.run_path,
        "config_path": rel(row.config_path) if row.config_path else None,
        "epochs": nested_get(cfg, "arguments.epochs", "args.epochs"),
        "batch_size_per_rank": nested_get(cfg, "batch_size_per_rank", "arguments.batch_size", "args.batch_size"),
        "global_train_batch_size": nested_get(cfg, "global_train_batch_size"),
        "lr": nested_get(cfg, "arguments.lr", "args.lr"),
        "weight_decay": nested_get(cfg, "arguments.weight_decay", "args.weight_decay"),
        "seed": nested_get(cfg, "arguments.seed", "args.seed"),
        "device": nested_get(cfg, "device", "arguments.device", "args.device"),
        "amp_enabled": nested_get(cfg, "amp_enabled", "arguments.amp", "args.amp"),
        "pretrained_weights": nested_get(cfg, "arguments.weights", "args.weights", "args.backbone_weights"),
        "image_size": nested_get(cfg, "args.image_size", "arguments.min_size"),
        "max_size": nested_get(cfg, "arguments.max_size", "args.image_size"),
        "keep_empty": nested_get(cfg, "arguments.keep_empty", "args.keep_empty"),
        "test_after_train": nested_get(cfg, "arguments.test_after_train", "args.test_after_train"),
    }


CONFIG_DF = pd.DataFrame([config_row(row) for row in RUNS.itertuples(index=False)]) if not RUNS.empty else pd.DataFrame()
if CONFIG_DF.empty:
    print("No config available.")
else:
    display(CONFIG_DF)

checks: list[dict[str, Any]] = []
if not CONFIG_DF.empty:
    comparable_cols = [
        "epochs",
        "batch_size_per_rank",
        "global_train_batch_size",
        "lr",
        "weight_decay",
        "seed",
        "pretrained_weights",
        "image_size",
        "max_size",
        "keep_empty",
        "test_after_train",
    ]
    for column in comparable_cols:
        values = CONFIG_DF[["run", column]].dropna()
        unique_values = values[column].astype(str).unique().tolist() if not values.empty else []
        checks.append(
            {
                "check": column,
                "status": "MATCH" if len(unique_values) <= 1 else "DIFF",
                "values": "; ".join(f"{r.run}={getattr(r, column)}" for r in CONFIG_DF.itertuples(index=False)),
            }
        )

if not RUN_INVENTORY.empty:
    for row in RUN_INVENTORY.itertuples(index=False):
        if not row.has_test_metrics:
            checks.append({"check": "test_metrics", "status": "MISSING", "values": f"{row.label}: no test_metrics.json"})
        if not row.has_config:
            checks.append({"check": "config", "status": "MISSING", "values": f"{row.label}: no config json"})

CHECKS_DF = pd.DataFrame(checks)
if CHECKS_DF.empty:
    print("No fairness checks available.")
else:
    display(CHECKS_DF)

## 5. Đường cong huấn luyện và validation

Các biểu đồ được lưu vào `notebooks/compare_training_results_figures/` để có thể mở lại hoặc đưa vào báo cáo.

In [ ]:
def plot_metric_curves(metrics_df: pd.DataFrame) -> Path | None:
    if metrics_df.empty:
        print("No metrics to plot.")
        return None

    plots = [
        ("train_loss_total", "Train loss", "lower is better"),
        ("valid_loss_total", "Validation loss", "lower is better"),
        ("map50", "Validation mAP50", "higher is better"),
        ("map_50_95", "Validation mAP50-95", "higher is better"),
        ("ar_100", "Validation AR@100", "higher is better"),
        ("pred_total", "Validation predictions", "diagnostic"),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(18, 9), constrained_layout=True)
    axes = axes.ravel()

    for ax, (column, title, subtitle) in zip(axes, plots):
        for run, df in metrics_df.groupby("run", sort=False):
            y = pd.to_numeric(df[column], errors="coerce") if column in df.columns else pd.Series(dtype="float64")
            if y.notna().any():
                ax.plot(df["epoch"], y, marker="o", linewidth=2, label=run)
        ax.set_title(f"{title}\n{subtitle}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(column)
        ax.legend(loc="best", fontsize=9)

    path = REPORT_DIR / "training_curves.png"
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    return path


training_curves_path = plot_metric_curves(METRICS_DF)
if training_curves_path:
    print(f"Saved: {training_curves_path}")

## 6. So sánh test metrics

Đây là bảng/biểu đồ quan trọng nhất để báo cáo kết quả sau khi đã train lại hai mô hình với điều kiện tương đương.

In [ ]:
def plot_test_bars(summary_df: pd.DataFrame) -> Path | None:
    metrics = ["test_map50", "test_map_50_95", "test_map75", "test_ar_100"]
    available = [m for m in metrics if m in summary_df.columns and summary_df[m].notna().any()]
    if summary_df.empty or not available:
        print("No test metrics to plot.")
        return None

    plot_df = summary_df.set_index("run")[available].apply(pd.to_numeric, errors="coerce")
    ax = plot_df.plot(kind="bar", figsize=(11, 5), width=0.78)
    ax.set_title("Test split metrics")
    ax.set_ylabel("Score")
    ax.set_xlabel("")
    ax.set_ylim(0, max(1.0, float(np.nanmax(plot_df.to_numpy())) * 1.15))
    ax.legend(loc="best")
    plt.xticks(rotation=0, ha="center")
    plt.tight_layout()

    path = REPORT_DIR / "test_metrics_bar.png"
    ax.figure.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    return path


if not SUMMARY_DF.empty:
    display(SUMMARY_DF[[c for c in ["run", "test_map50", "test_map_50_95", "test_map75", "test_ar_100", "test_pred_total", "test_gt_total"] if c in SUMMARY_DF.columns]])

test_bar_path = plot_test_bars(SUMMARY_DF)
if test_bar_path:
    print(f"Saved: {test_bar_path}")

## 7. AP50 theo từng lớp

Biểu đồ này giúp phát hiện mô hình thắng/thua vì một vài lớp cụ thể hay cải thiện đồng đều. Nếu artifact thiếu AP theo lớp, cell sẽ bỏ qua phần đó.

In [ ]:
def collect_class_ap50(summary_df: pd.DataFrame, metrics_df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []

    if not summary_df.empty:
        class_cols = [c for c in summary_df.columns if c.startswith("test_class_") and c.endswith("_ap50")]
        for row in summary_df.itertuples(index=False):
            for column in class_cols:
                value = getattr(row, column)
                if pd.notna(value):
                    rows.append({"run": row.run, "source": "test", "class": column.removeprefix("test_class_").removesuffix("_ap50"), "ap50": value})

    if not rows and not metrics_df.empty:
        class_cols = [c for c in metrics_df.columns if c.startswith("class_") and c.endswith("_ap50")]
        best_lookup = BEST_EPOCH_DF.set_index("run")["best_epoch"].to_dict() if not BEST_EPOCH_DF.empty else {}
        for run, df in metrics_df.groupby("run", sort=False):
            best_epoch = best_lookup.get(run, df["epoch"].max())
            best_df = df[df["epoch"] == best_epoch]
            if best_df.empty:
                continue
            row = best_df.iloc[0]
            for column in class_cols:
                value = row[column]
                if pd.notna(value):
                    rows.append({"run": run, "source": "validation_best_epoch", "class": column.removeprefix("class_").removesuffix("_ap50"), "ap50": value})

    return pd.DataFrame(rows)


CLASS_AP50_DF = collect_class_ap50(SUMMARY_DF, METRICS_DF)
if CLASS_AP50_DF.empty:
    print("No per-class AP50 metrics available.")
else:
    pivot = CLASS_AP50_DF.pivot_table(index="class", columns="run", values="ap50", aggfunc="first").sort_index()
    display(pivot)
    ax = pivot.plot(kind="bar", figsize=(13, 5), width=0.78)
    ax.set_title("Per-class AP50")
    ax.set_ylabel("AP50")
    ax.set_xlabel("Class")
    ax.set_ylim(0, max(1.0, float(np.nanmax(pivot.to_numpy())) * 1.15))
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    class_path = REPORT_DIR / "class_ap50_bar.png"
    ax.figure.savefig(class_path, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"Saved: {class_path}")

## 8. Kết luận nhanh từ artifact hiện tại

Cell cuối chỉ tóm tắt chênh lệch số học từ file hiện có. Nếu bạn vừa sửa logic train, hãy train lại cả hai model trước khi dùng kết luận này trong báo cáo.

In [ ]:
def metric_delta_table(summary_df: pd.DataFrame) -> pd.DataFrame:
    if summary_df.empty or len(summary_df) < 2:
        return pd.DataFrame()

    metrics = ["test_map50", "test_map_50_95", "test_map75", "test_ar_100"]
    rows: list[dict[str, Any]] = []
    base = summary_df.iloc[0]
    for other_idx in range(1, len(summary_df)):
        other = summary_df.iloc[other_idx]
        for metric in metrics:
            if metric not in summary_df.columns:
                continue
            base_value = pd.to_numeric(pd.Series([base.get(metric)]), errors="coerce").iloc[0]
            other_value = pd.to_numeric(pd.Series([other.get(metric)]), errors="coerce").iloc[0]
            if pd.isna(base_value) or pd.isna(other_value):
                continue
            rows.append(
                {
                    "comparison": f"{other['run']} - {base['run']}",
                    "metric": metric,
                    "baseline_value": base_value,
                    "candidate_value": other_value,
                    "delta_abs": other_value - base_value,
                    "delta_pct_of_baseline": np.nan if base_value == 0 else (other_value - base_value) / abs(base_value) * 100,
                }
            )
    return pd.DataFrame(rows)


DELTA_DF = metric_delta_table(SUMMARY_DF)
if DELTA_DF.empty:
    print("Need at least two runs with comparable test metrics for delta summary.")
else:
    display(DELTA_DF)
    delta_path = REPORT_DIR / "metric_deltas.csv"
    DELTA_DF.to_csv(delta_path, index=False)
    print(f"Saved: {delta_path}")

if not CHECKS_DF.empty and (CHECKS_DF["status"] != "MATCH").any():
    print("Fairness warnings exist. Review CHECKS_DF before making a final claim.")
else:
    print("No fairness warning detected by the notebook checks.")